# Baseline: TF-IDF + logistic regression

Text-only and text-plus-position baselines trained on all 180,040 training sentences. CPU runtime, about a minute.

In [ ]:
# ===== STEP 2: baseline - TF-IDF + logistic regression (trained with SGD for speed), with and without sentence position =====
import os, time, json
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

DATA = "/content/drive/MyDrive/sci-classify/data"
RES = DATA.replace("/data", "/results"); os.makedirs(RES, exist_ok=True)
train = pd.read_parquet(f"{DATA}/train.parquet")
test = pd.read_parquet(f"{DATA}/test.parquet")
test_llm = pd.read_parquet(f"{DATA}/test_llm_1000.parquet")
LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]

def bootstrap_ci(y, p, n=1000, seed=0):
    """95% bootstrap interval for macro-F1."""
    rng = np.random.default_rng(seed); y, p = np.asarray(y), np.asarray(p); idx = np.arange(len(y))
    scores = [f1_score(y[s], p[s], average="macro") for s in (rng.choice(idx, len(idx)) for _ in range(n))]
    return np.percentile(scores, [2.5, 97.5])

def position_features(df):
    """Where the sentence sits in its abstract: relative position, plus first / last flags."""
    rel = df.line_number / (df.total_lines - 1).clip(lower=1)
    return csr_matrix(np.c_[rel, rel**2, (df.line_number == 0), (df.line_number == df.total_lines - 1)].astype(float))

def report(name, y, p, secs, n_test):
    lo, hi = bootstrap_ci(y, p)
    row = {"model": name, "eval_set": n_test, "accuracy": accuracy_score(y, p),
           "macro_f1": f1_score(y, p, average="macro"), "ci_low": lo, "ci_high": hi, "seconds": secs}
    print(f"{name:38s} {n_test:>11s}  acc {row['accuracy']:.3f}  macro-F1 {row['macro_f1']:.3f} "
          f"(95% CI {lo:.3f} to {hi:.3f})  {secs:.0f}s")
    return row

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
t0 = time.time(); Xtr = vec.fit_transform(train.text); Xte = vec.transform(test.text); Xll = vec.transform(test_llm.text)

rows, preds = [], {}
for name, with_pos in [("TF-IDF + logistic regression", False), ("TF-IDF + position + logistic regression", True)]:
    t1 = time.time()
    A, B, C = (Xtr, Xte, Xll) if not with_pos else (hstack([Xtr, position_features(train)]).tocsr(),
                                                    hstack([Xte, position_features(test)]).tocsr(),
                                                    hstack([Xll, position_features(test_llm)]).tocsr())
    clf = SGDClassifier(loss="log_loss", alpha=2e-6, max_iter=30, tol=1e-4, class_weight="balanced", random_state=42).fit(A, train.label)
    secs = time.time() - t1
    p_full, p_llm = clf.predict(B), clf.predict(C)
    rows.append(report(name, test.label, p_full, secs, "full test"))
    rows.append(report(name, test_llm.label, p_llm, secs, "1000 set"))
    preds[name] = p_llm.tolist()
    if with_pos:
        print("\nPer-class F1 (full test set, with position):")
        print(classification_report(test.label, p_full, labels=LABELS, digits=3))
        cm = pd.DataFrame(confusion_matrix(test.label, p_full, labels=LABELS), index=LABELS, columns=LABELS)
        print("Confusion matrix (rows = true label, columns = predicted):"); print(cm.to_string())

pd.DataFrame(rows).to_csv(f"{RES}/step2_baseline.csv", index=False)
json.dump({"ids": list(range(len(test_llm))), "gold": test_llm.label.tolist(), "preds": preds},
          open(f"{RES}/step2_preds_1000.json", "w"))
print(f"\nSaved results to {RES}")
